In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import ROOT as root
from ROOT import TH2F
from openpyxl import Workbook
import pytz
from datetime import datetime
import json
import pandas as pd
import statistics
from array import array
import glob
import re

Welcome to JupyROOT 6.30/04


In [3]:
n_wafer = 83 #
n_row = 6 #
n_col = 6 # 
n_sensor = 44
start_wafer = 1 #first wafer number in the dataset
start_row = 1 #the first row of the wafer to be measured
start_col = 1 #the first column of the wafer to be measured 
start_sensor = 1


dtz = datetime(2026, 5, 28, 12, 0, 0)
dtz = dtz.replace(tzinfo=pytz.utc)
dtz.astimezone(pytz.timezone("Europe/Rome"))

user = "fsiviero" # cern user of who's uploading the test
location = "Torino" # where the test was performed


wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_16x16_PRE-SERIES.xlsx') # excel that maps 'serial number --> vendor, batch, wafer, row, column'

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'

vendor_file ="/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_16x16_PRE-SERIES_Vendor_IV.json"

In [5]:
optical_inspection_passed = bool()
optical_inspection_json = []

sensor_list = [41,38,34,21,6]

for j in range(n_sensor):
    if (j+start_sensor) not in sensor_list :
            print(j+start_sensor)
            opt = {'component': str(wb[(wb['Wafer'] == 17) & (wb['Sensor Number'] == j+start_sensor) ]['SerialNumber'].iloc[0]),
                      'name': 'Optical Inspection',
                      'measurement_date': dtz.isoformat(), #year, month, day, hour, minute, second
                      'location': location,
                      'user_created': user,
                      'version': 'v0',
                      'passed': True,
                      'comment': None
                }
            optical_inspection_json.append(opt)


with open(save_path+"FBK_Torino_optical_inspection_W17.json", 'w') as f:
   json.dump(optical_inspection_json, f, indent=4)

1
2
3
4
5
7
8
9
10
11
12
13
14
15
16
17
18
19
20
22
23
24
25
26
27
28
29
30
31
32
33
35
36
37
39
40
42
43
44


In [3]:
optical_inspection_passed = bool()
optical_inspection_json = []

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_16x16_PRE-SERIES.xlsx')
file_qa = root.TFile.Open("/Users/icosivi/Desktop/PRE-SERIE/FBK/PRE-SERIE_FBK_Vendor_16x16_IV.root")
tree_qa = file_qa.Get("Tree")

dtz = datetime(2026, 5, 9, 12, 0, 0)
dtz = dtz.replace(tzinfo=pytz.utc)
dtz.astimezone(pytz.timezone("Europe/Rome"))

user = "fsiviero" # cern user of who's uploading the test
location = "FBK" # where the test was performed

for event in tree_qa:
  if event.fbk_optical_inspection == 0:
    optical_inspection_passed = True
  else:
    optical_inspection_passed = False
  
  opt = {'component': str(wb[(wb['Wafer'] == event.wafer) & (wb['Row'] == event.row) & (wb['Column'] == event.column)]['SerialNumber'].iloc[0]),
                      'name': 'Optical Inspection',
                      'measurement_date': dtz.isoformat(), #year, month, day, hour, minute, second
                      'location': location,
                      'user_created': user,
                      'version': 'v0',
                      'passed': optical_inspection_passed,
                      'comment': None
  }
  optical_inspection_json.append(opt)


with open(save_path+"FBK_optical_inspection.json", 'w') as f:
   json.dump(optical_inspection_json, f, indent=4)